# TD02 — Réseau mono-couche (3 perceptrons)

## Objectif

Passer d'un neurone (TD01) à une **couche de 3 perceptrons** : chaque neurone $k$ calcule un score $z_k = W_k \cdot x + b_k$, et le réseau renvoie l'indice du score maximal $y = \mathrm{argmax}(z_1, z_2, ..., z_n)$. On l'apprend avec la **règle de Rosenblatt multi-classes**, puis on l'applique en 2D et en 20D (lettres S/V/X).


**Notes - Écart au modèle du cours**
1. Dans le cours, on associait un facteur `-1` au biais. Ici on associe un facteur `1` ( donc on a la sortie `z = W \cdot x + b` au lieu de `z = W \cdot x - b`). La seule différence est au niveau de l'implémentation, le réseau de neurones se comporte exactement de la même manière.
2. Dans le cours, la fonction d'activation (ex. Heaviside) s'applique à la sortie d'**un seul** neurone : $y = H(w \cdot x + b)$. Ici, on regroupe $K$ neurones et la décision se fait par un `argmax` appliqué au **vecteur** des scores, c'est-à-dire à la sortie de **plusieurs** neurones : $y = \mathrm{argmax}(W x + b)$.


**Exemple**: entrée de dimension deux, avec une couche de 3 neurones:
![image.png](assets/02_mono_couche-image-2.png)

## Préambule : l'opérateur `argmax`

`argmax` renvoie l'**indice** (la position) de la plus grande valeur d'un vecteur — pas la valeur elle-même :

**Exemples**
- $\mathrm{argmax}([0.2,\; -0.5,\; 0.7]) = 2$ (le maximum $0.7$ est en position 2)
- $\mathrm{argmax}([-1.0,\; 3.0,\; 2.0]) = 1$
- $\mathrm{argmax}([0.0,\; 0.0,\; 0.0]) = 0$ (en cas d'égalité, `np.argmax` renvoie le premier indice)

En Python : `np.argmax([0.2, -0.5, 0.7])` renvoie `2`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants
VecInt = np.ndarray[tuple[int], np.dtype[np.int64]]       # vecteur 1D d'entiers

## 1. Étude en 2D

Premier cas d'étude, en dimension 2 : trois classes gaussiennes (jeu 1 séparable, jeu 2 piégé au centre).



### 1.1 Génération de jeux de données 2D

On génère deux jeux de données: `(X1, t1)` et `(X2, t2)`

In [ ]:
def gen_jeu1(n=25, sigma=0.6, seed=0):
    rng = np.random.default_rng(seed)
    centres = [(-2, -2), (2, -2), (0, 2)]
    X_list, t_list = [], []
    for k, (cx, cy) in enumerate(centres):
        pts = rng.normal(loc=(cx, cy), scale=sigma, size=(n, 2))
        X_list.append(pts)
        t_list += [k] * n
    return np.vstack(X_list), np.array(t_list)


def gen_jeu2(n=15, seed=1):
    rng = np.random.default_rng(seed)
    Xa = rng.normal(loc=(0, 0), scale=0.6, size=(n, 2))
    Xb = rng.normal(loc=(0, 0), scale=0.6, size=(n, 2))
    X = np.vstack([Xa, Xb])
    t = np.array([0] * n + [1] * n)
    return X, t

In [ ]:
X1, t1 = gen_jeu1()
_X2, _t2 = gen_jeu2()
X2 = np.vstack([X1, _X2])
t2 = np.concatenate([t1, _t2])

### 1.2 Visualisation des données

Les cellules ci-dessous affichent les deux jeux : `X1, t1` (jeu 1) et `X2, t2` (jeu 2).

In [ ]:
def display_data(X, t, titre=None):
    couleurs = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
    fig, ax = plt.subplots()
    for k in sorted(set(t)):
        ax.scatter(
            X[t == k, 0], X[t == k, 1],
            c=couleurs[k % len(couleurs)],
            edgecolors='k', s=25, label=f"classe {k}"
        )
    ax.set_aspect('equal')
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    if titre:
        ax.set_title(f"{titre} ({len(X)} pts)")
    ax.legend()
    ax.grid(True)
    plt.show()

In [ ]:
display_data(X1, t1, titre="Jeu 1 : séparable")
display_data(X2, t2, titre="Jeu 2 : piège central")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1</legend>

Comparez les deux jeux de données : qu'est-ce qui les distingue ?

</fieldset>

*Votre réponse :*

# TOCLEAR

Le jeu 1 est formé de trois nuages gaussiens bien séparés (un par classe) : il est **linéairement séparable**.

Le jeu 2 reprend le jeu 1 et y ajoute, **au centre**, des points des classes 0 et 1 tirés selon la même loi : ils sont mélangés, aucune frontière (linéaire ou non) ne peut les séparer parfaitement. Le jeu 2 n'est pas linéairement séparable.

### 1.3 Modèle : couche mono-couche `y = argmax(W·x + b)`

Une couche de $K$ perceptrons : $W$ de forme $(K \times d)$, $b$ de forme $(K,)$. La règle de Rosenblatt multi-classes : si la prédiction $p$ diffère de la cible $t$, on **renforce** le bon neurone $t$ (faux négatif) et on **punit** le mauvais gagnant $p$ (faux positif).

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Complétez la classe `ReseauMonoCouche` :

1. `__init__(d, K)` : initialise `W` (forme `(K, d)`) et `b` (forme `(K,)`) à zéro ;
2. `predict(x)` : calcule `h = W·x + b` et renvoie l'indice du score maximal (`np.argmax`) ;
3. `learn_one(x, t, r=0.1)` : si la prédiction `p` diffère de la cible `t`, on **renforce** le bon neurone `t` (`W[t] += r·x`, `b[t] += r`) et on **punit** le mauvais gagnant `p` (`W[p] -= r·x`, `b[p] -= r`) ;
4. `train(X, t, epochs=50)` : boucle sur les époques (au plus `epochs`), applique `learn_one` à chaque point, compte les erreurs, et s'arrête dès qu'une époque ne fait plus aucune erreur.

</fieldset>

In [ ]:
class ReseauMonoCouche:
    def __init__(self, d: int, K: int):
        # BEGIN
        # raise NotImplementedError
        ## Indice : W de forme (K, d) et b de forme (K,), initialisés à zéro
        self.W = np.zeros((K, d))
        self.b = np.zeros(K)
        # END

    def predict(self, x: Vec) -> int:
        # BEGIN
        # raise NotImplementedError
        h = self.W @ x + self.b
        return int(np.argmax(h))
        # END

    def learn_one(self, x: Vec, t: int, r: float = 0.1) -> None:
        # BEGIN
        # raise NotImplementedError
        ## Indice : p = self.predict(x) ; si p != t : W[t]+=r*x, b[t]+=r, W[p]-=r*x, b[p]-=r
        p = self.predict(x)
        if p != t:
            self.W[t] += r * x
            self.b[t] += r
            self.W[p] -= r * x
            self.b[p] -= r
        # END

    def train(self, X: Mat, t: VecInt, epochs: int = 50) -> 'ReseauMonoCouche':
        # BEGIN
        # raise NotImplementedError
        ## Indice : pour chaque époque, compte les erreurs et arrête si err == 0
        for _ in range(epochs):
            err = 0
            for xi, ti in zip(X, t):
                if self.predict(xi) != int(ti):
                    err += 1
                self.learn_one(xi, int(ti))
            if err == 0:
                break
        return self
        # END

#### Vérification de `ReseauMonoCouche`

On teste successivement :
- `predict` : le calcul `h = W·x + b` et l'`argmax` sur des scores choisis ;
- `learn_one` : une étape d'apprentissage doit exactement renforcer `t` et punir `p` ;
- `train` : sur deux points, la simulation à la main donne les poids finaux exacts.

In [ ]:
def check_predict():
    c = ReseauMonoCouche(d=2, K=3)
    c.W = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
    c.b = np.zeros(3)
    assert c.predict(np.array([1.0, 0.0])) == 0
    assert c.predict(np.array([0.0, 1.0])) == 1
    assert c.predict(np.array([-1.0, -1.0])) == 2
    print("predict check — OK")


def check_learn_one():
    c = ReseauMonoCouche(d=2, K=3)
    # predict([1,1]) = argmax([0,0,0]) = 0 ; cible 1 -> erreur
    c.learn_one(np.array([1.0, 1.0]), 1, r=0.1)
    assert np.array_equal(c.W[1], np.array([0.1, 0.1])), c.W[1]
    assert c.b[1] == 0.1, c.b[1]
    assert np.array_equal(c.W[0], np.array([-0.1, -0.1])), c.W[0]
    assert c.b[0] == -0.1, c.b[0]
    assert np.array_equal(c.W[2], np.array([0.0, 0.0]))
    print("learn_one check — OK")


def check_train():
    c = ReseauMonoCouche(d=2, K=2)
    X = np.array([[1.0, 0.0], [0.0, 1.0]])
    t = np.array([1, 0])
    c.train(X, t)
    # simulation à la main (r = 0.1) :
    #   époque 1 : x=[1,0], t=1 : h=[0,0] -> p=0 != 1 -> W=[[-0.1,0],[0.1,0]], b=[-0.1,0.1]
    #              x=[0,1], t=0 : h=[-0.1,0.1] -> p=1 != 0 -> W=[[-0.1,0.1],[0.1,-0.1]], b=[0,0]
    #   époque 2 : x=[1,0] : h=[-0.1,0.1] -> 1 ok ; x=[0,1] : h=[0.1,-0.1] -> 0 ok -> arrêt
    assert np.array_equal(c.W, np.array([[-0.1, 0.1], [0.1, -0.1]])), c.W
    assert np.array_equal(c.b, np.array([0.0, 0.0])), c.b
    print("train check — OK")


check_predict()
check_learn_one()
check_train()

### 1.4 Entraînement : jeu 1 vs jeu 2

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3a</legend>

Complétez `accuracy` : pourcentage (entre 0 et 100) de bonnes prédictions du réseau sur `X`.

</fieldset>

In [ ]:
def accuracy(reseau: ReseauMonoCouche, X: Mat, t: VecInt) -> float:
    """Pourcentage de bonnes prédictions (sur les données d'entraînement)."""
    # BEGIN
    # raise NotImplementedError
    pred = np.array([reseau.predict(x) for x in X])
    return (pred == t).mean() * 100
    # END

In [ ]:
def check_accuracy():
    c = ReseauMonoCouche(d=2, K=3)
    c.W = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0]])
    c.b = np.zeros(3)
    X = np.array([[1.0, 0.0], [0.0, 1.0], [-1.0, -1.0], [2.0, 0.0]])
    assert accuracy(c, X, np.array([0, 1, 2, 0])) == 100.0
    assert accuracy(c, X, np.array([0, 1, 2, 1])) == 75.0
    print("accuracy check — OK")


check_accuracy()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3b</legend>

Entraînez deux couches identiques (`d=2`, `K=3`, paramètres de `train` par défaut) : `couche1` sur `X1, t1` et `couche2` sur `X2, t2`. Stockez leurs accuracies (sur leur propre jeu d'entraînement) dans `acc_jeu_1` et `acc_jeu_2`.

</fieldset>

In [ ]:
# BEGIN
# couche1 = ...
# acc_jeu_1 = ...
# couche2 = ...
# acc_jeu_2 = ...
couche1 = ReseauMonoCouche(d=2, K=3)
couche1.train(X1, t1)
acc_jeu_1 = accuracy(couche1, X1, t1)
couche2 = ReseauMonoCouche(d=2, K=3)
couche2.train(X2, t2)
acc_jeu_2 = accuracy(couche2, X2, t2)
# END

In [ ]:
def check_couches():
    for name, c, acc, X, t in [("1", couche1, acc_jeu_1, X1, t1), ("2", couche2, acc_jeu_2, X2, t2)]:
        ref = ReseauMonoCouche(d=2, K=3).train(X, t)
        assert np.array_equal(c.W, ref.W) and np.array_equal(c.b, ref.b), f"couche{name} : poids inattendus"
        assert acc == accuracy(ref, X, t), f"acc_jeu_{name} incorrecte"
    print("entraînement check — OK")


check_couches()

Après la vérification, la cellule suivante affiche les deux accuracies.

In [ ]:
print(f"accuracy jeu 1 : {acc_jeu_1:.1f}%")
print(f"accuracy jeu 2 : {acc_jeu_2:.1f}%")

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3c</legend>

Comparez les deux accuracies : que se passe-t-il ?

</fieldset>

*Votre réponse :*

# TOCLEAR

Sur le jeu 1, la couche atteint 100 % : les données sont linéairement séparables, la règle de Rosenblatt converge et `train` s'arrête avant `epochs`.

Sur le jeu 2, l'accuracy plafonne (environ 74 %) : les points centraux des classes 0 et 1 sont mélangés, aucun `W` ne les sépare. L'apprentissage oscille jusqu'à la dernière époque, et les poids finaux dépendent de l'endroit où l'on s'arrête.

#### Frontières de décision

`plot_frontieres` affiche :
- **fond coloré** = vraie décision `argmax_k h_k` (régions convexes, bords = `h_i == h_j`) ;
- **pointillés noirs** = zéros de chaque score `h_k = 0` (où le neurone `k` bascule). **Ce N'EST PAS la frontière de décision !**

In [ ]:
def plot_frontieres(ax, X, t, couche, lim=(-4, 4)):
    xx, yy = np.meshgrid(np.linspace(lim[0], lim[1], 300), np.linspace(lim[0], lim[1], 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = np.argmax(grid @ couche.W.T + couche.b, axis=1).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5],
                colors=['tab:blue', 'tab:orange', 'tab:green'], alpha=0.12)
    for k, c in enumerate(['tab:blue', 'tab:orange', 'tab:green']):
        ax.scatter(X[t == k, 0], X[t == k, 1], c=c, edgecolors='k', s=25, label=f"classe {k}")
    for k in range(3):
        H = (grid @ couche.W[k] + couche.b[k]).reshape(xx.shape)
        ax.contour(xx, yy, H, levels=[0], colors='k', linewidths=1.5, linestyles='--')
    ax.plot([], [], 'k--', label='h_k=0 (indicatif, pas frontière)')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_frontieres(axes[0], X1, t1, couche1)
axes[0].set_title("Jeu 1")
plot_frontieres(axes[1], X2, t2, couche2)
axes[1].set_title("Jeu 2")
plt.tight_layout()
plt.show()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4</legend>

Lisez la figure :

1. Trouvez un pointillé qui traverse une zone de couleur uniforme. Que concluez-vous ?
2. Comparez les résultats jeu 1 vs jeu 2.

</fieldset>

*Votre réponse :*

# TOCLEAR

1. De part et d'autre de ce pointillé, le score $h_k$ change de signe mais la décision ne change pas : la décision ne dépend pas du signe de chaque score, seulement de **quel score est le plus grand**. Un neurone peut gagner avec un score négatif (si les autres sont encore plus négatifs). Les frontières de décision sont les lieux $h_i = h_j$, pas $h_k = 0$.
2. Jeu 1 : chaque nuage est contenu dans sa région, tout est bien classé. Jeu 2 : les régions restent convexes (le modèle est linéaire), elles ne peuvent donc pas isoler les points mélangés au centre ; une partie des points des classes 0 et 1 tombe dans la mauvaise région.

## 2. Étude en 20D : lettres S/V/X en 5×4 (20 dimensions)

Même classe `ReseauMonoCouche`, réutilisée en dimension 20 : 3 templates `TPL` encodés en ±1, bruités par `n_flip` pixels inversés. Objectif : montrer que le même modèle fonctionne en 20D quand les données sont séparables, et lire chaque ligne de `W` comme un prototype.

### 2.1 Génération des lettres bruitées

3 templates `TPL` encodés en ±1, bruités en inversant `n_flip` pixels au hasard.

In [ ]:
TPL = {
    0: np.array([0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0]),
    1: np.array([1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0]),
    2: np.array([1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1]),
}


def enc(v):
    return np.where(v == 1, 1.0, -1.0)


def gen_lettres(n_per=10, n_flip=2, seed=2):
    rng = np.random.default_rng(seed)
    X, t = [], []
    for k in range(3):
        base = enc(TPL[k])
        for _ in range(n_per):
            w = base.copy()
            idx = rng.choice(20, size=n_flip, replace=False)
            w[idx] *= -1
            X.append(w)
            t.append(k)
    return np.array(X), np.array(t)

In [ ]:
def affiche_lettre(lettre: Vec, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.imshow(lettre.reshape(5, 4), cmap="RdBu", vmin=-1, vmax=1)


affiche_lettre(TPL[0])

### 2.2 Entraînement (2 flips)

On entraîne avec un peu de bruit (2 flips) pour obtenir une résistance au bruit.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5</legend>

Créez une couche `c_let` en dimension 20 à 3 neurones, et entraînez-la (paramètres de `train` par défaut) sur les lettres `X_let, t_let`.

</fieldset>

In [ ]:
X_let, t_let = gen_lettres()  # 30 lettres : 10 par classe, bruitées à 2 flips

# BEGIN
# c_let = ...
c_let = ReseauMonoCouche(d=20, K=3)
c_let.train(X_let, t_let)
# END

In [ ]:
def check_c_let():
    ref = ReseauMonoCouche(d=20, K=3).train(X_let, t_let)
    assert c_let.W.shape == (3, 20), f"W de forme (3, 20) attendue, obtenu {c_let.W.shape}"
    assert np.array_equal(c_let.W, ref.W) and np.array_equal(c_let.b, ref.b), "poids inattendus"
    print("c_let check — OK")


check_c_let()

Après la vérification, la cellule suivante affiche l'accuracy sur les lettres d'entraînement.

In [ ]:
print(f"accuracy sur le train : {accuracy(c_let, X_let, t_let):.1f}%")

### 2.3 Test à 5 flips

On génère 30 lettres **nouvelles**, bruitées à 5 flips : `X_let_test, t_let_test`.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6</legend>

Calculez l'accuracy (en %) du réseau `c_let` sur ces lettres de test, stockée dans `acc_let_test`.

</fieldset>

In [ ]:
X_let_test, t_let_test = gen_lettres(n_per=10, n_flip=5, seed=99)

# BEGIN
# acc_let_test = ...
acc_let_test = accuracy(c_let, X_let_test, t_let_test)
# END

In [ ]:
pred_ref = np.array([c_let.predict(x) for x in X_let_test])
assert acc_let_test == (pred_ref == t_let_test).mean() * 100, "acc_let_test incorrecte"
print("acc_let_test check — OK")

In [ ]:
print(f"accuracy lettres test (5 flips) : {acc_let_test:.1f}%")

### 2.4 Vérification à la main

On vérifie sur la première lettre de test, `X_let_test[0]`, que le réseau l'a reconnue en regardant les scores $h = W \cdot x + b$ de chaque neurone (le plus grand `h_k` donne la prédiction).

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 7</legend>

Calculez le vecteur des scores `h` des 3 neurones pour la lettre `X_let_test[0]`, sans passer par `predict`.

</fieldset>

In [ ]:
# BEGIN
# h = ...
h = c_let.W @ X_let_test[0] + c_let.b
# END

In [ ]:
assert np.shape(h) == (3,), f"h doit contenir 3 scores, obtenu {np.shape(h)}"
assert np.allclose(h, c_let.W @ X_let_test[0] + c_let.b), "h incorrect"
assert int(np.argmax(h)) == c_let.predict(X_let_test[0])
print("scores check — OK")

Après la vérification, la cellule suivante affiche les scores, la prédiction et la lettre ; vérifiez que la lettre affichée correspond à la prédiction (0 = S, 1 = V, 2 = X).

In [ ]:
print("scores :", h)
print("prédiction :", int(np.argmax(h)), "— cible :", t_let_test[0])
affiche_lettre(X_let_test[0])

### 2.5 Lire `W` : quel prototype pour quelle lettre ?

Chaque ligne de `W` (20 nombres) ré-affichée en 5×4 (`RdBu`, rouge = +1, bleu = −1) est un prototype appris. On affiche ci-dessous les templates (en haut) et les lignes de `W` (en bas).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for k in range(3):
    axes[0, k].imshow(enc(TPL[k]).reshape(5, 4), cmap="RdBu", vmin=-1, vmax=1)
    axes[0, k].set_title(f"template {'SVX'[k]}")
    axes[0, k].axis("off")
    axes[1, k].imshow(c_let.W[k].reshape(5, 4), cmap="RdBu", vmin=-1, vmax=1)
    axes[1, k].set_title(f"W classe {k} -> lettre ?")
    axes[1, k].axis("off")
plt.suptitle("haut = templates, bas = poids appris (associez !)")
plt.tight_layout()
plt.show()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 8</legend>

Associez chaque ligne de `W` à sa lettre. Que remarque-t-on ?

</fieldset>

*Votre réponse :*

# TOCLEAR

La ligne `W[k]` ressemble au template de la lettre `k` (S, V, X) : les pixels allumés de la lettre ont des poids positifs, les pixels éteints des poids négatifs. Chaque neurone calcule donc une **corrélation** entre l'entrée et son prototype.

Les pixels identiques dans les trois lettres ont des poids nuls ou faibles : ils ne servent pas à distinguer les classes. (Chaque mise à jour ajoute `r·x` à une ligne et le retire d'une autre : la somme des lignes de `W` reste nulle.)

### 2.6 Robustesse au bruit

On fait varier `n_flip` de 0 à 8 (train fixe à 2 flips) et on mesure l'accuracy sur un jeu de test.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 9</legend>

Remplissez la liste `accs` : pour chaque `nf` de `flips`, générez un jeu de test `gen_lettres(n_per=10, n_flip=nf, seed=100 + nf)` et ajoutez l'accuracy (en %) de `c_let` sur ce jeu.

</fieldset>

In [ ]:
flips = list(range(0, 9))
accs = []
# BEGIN
# for nf in flips:
#     accs.append(...)
for nf in flips:
    Xt, tt = gen_lettres(n_per=10, n_flip=nf, seed=100 + nf)
    accs.append(accuracy(c_let, Xt, tt))
# END

In [ ]:
def check_accs():
    assert len(accs) == len(flips), f"{len(flips)} valeurs attendues, obtenu {len(accs)}"
    for nf, a in zip(flips, accs):
        Xt, tt = gen_lettres(n_per=10, n_flip=nf, seed=100 + nf)
        assert a == accuracy(c_let, Xt, tt), f"accuracy incorrecte pour n_flip={nf}"
    print("accs check — OK")


check_accs()

Après la vérification, la cellule suivante trace la courbe `accuracy(n_flip)`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(flips, accs, marker='o')
ax.axhline(80, color='gray', linestyle='--')
ax.set_xlabel("n_flip (pixels inversés)")
ax.set_ylabel("accuracy test %")
ax.set_title("Robustesse au bruit (train flip=2)")
ax.grid(True)
plt.show()
print(list(zip(flips, [f'{a:.1f}%' for a in accs])))

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 10</legend>

À partir de quel `n_flip` passe-t-on sous 80 % ? Pourquoi ? (distance au prototype vs bruit)

</fieldset>

*Votre réponse :*

# TOCLEAR

On passe sous 80 % à partir de `n_flip = 7` (100 % jusqu'à 3 flips, puis une baisse progressive).

Deux templates diffèrent sur 12 pixels. Une lettre bruitée à `n_flip` pixels est à distance `n_flip` de son prototype, mais peut se rapprocher d'un autre prototype si les pixels inversés font partie de ceux qui distinguent les deux lettres. Au-delà d'environ 6 flips (la moitié de 12), une lettre peut se retrouver plus proche d'un autre prototype que du sien : le réseau, qui mesure une corrélation avec chaque prototype, se trompe de plus en plus souvent.

## Synthèse

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 11</legend>

Pourquoi le même modèle échoue-t-il sur le jeu 2 (2D) mais réussit-il en 20D sur les lettres ?

</fieldset>

*Votre réponse :*

# TOCLEAR

Ce n'est pas la dimension qui compte, c'est la **séparabilité linéaire**. Les lettres forment trois nuages bien séparés autour des prototypes (±1 + quelques flips) → Rosenblatt converge. Le jeu 2 a ses classes 0/1 **mélangées au centre** → aucun `W` linéaire ne marche, il faut du non-linéaire / multi-couches (TD03).

## Conclusion
- Séparable → Rosenblatt converge, régions convexes (fond `argmax`).
- `h_k = 0` (pointillés) ≠ frontière : la frontière est `h_i == h_j`.
- Piège central → oscillation, < 100 % : limite du linéaire.
- Lettres 20D → OK car séparables ; `W` lisible = prototypes ; robustesse chute avec le bruit.
- Suite : besoin de non-linéarité / multi-couches (TD03).